# Colab: Evaluate Trained Virchow on PCam Test (Beginner)

This notebook runs **test-only evaluation** using:
- your trained run folder (`model_best.pt`, `temperature_fit.json`, etc.), and
- your preprocessed PCam folder (`test_x.h5`, `test_y.h5`).

It uses: `scripts/evaluate_virchow_preprocessed_test_colab.py`.

## You only edit 4 values
- `REPO_DIR`
- `DRIVE_RUN_DIR`
- `DRIVE_PREPROCESSED_DIR`
- `HF_TOKEN` (if Virchow is gated)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi


If your repo is not already present in Colab, run this clone cell.
If your repo is in Drive already, skip and set `REPO_DIR` in the next cell to that Drive path.


In [ ]:
%cd /content
!git clone https://github.com/<your-username>/<your-repo>.git GP_ECG
%cd /content/GP_ECG


In [ ]:
# ==== EDIT THESE 4 LINES ====
REPO_DIR = '/content/GP_ECG'
DRIVE_RUN_DIR = '/content/drive/MyDrive/GP_ECG_RUNS/virchow_macenko_bench_run_01'
DRIVE_PREPROCESSED_DIR = '/content/drive/MyDrive/GP_ECG_DATA/preprocessed_macenko_benchmark_style'
HF_TOKEN = ''

# Optional speed: copy from Drive -> local SSD
COPY_TO_LOCAL_SSD = True

import os
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN

print('REPO_DIR:', REPO_DIR)
print('DRIVE_RUN_DIR:', DRIVE_RUN_DIR)
print('DRIVE_PREPROCESSED_DIR:', DRIVE_PREPROCESSED_DIR)


In [ ]:
import os
import shutil

if COPY_TO_LOCAL_SSD:
    LOCAL_RUN_DIR = '/content/local_virchow_run'
    LOCAL_PREPROCESSED_DIR = '/content/local_preprocessed_test'

    if not os.path.isdir(DRIVE_RUN_DIR):
        raise FileNotFoundError('DRIVE_RUN_DIR not found: ' + DRIVE_RUN_DIR)
    if not os.path.isdir(DRIVE_PREPROCESSED_DIR):
        raise FileNotFoundError('DRIVE_PREPROCESSED_DIR not found: ' + DRIVE_PREPROCESSED_DIR)

    if os.path.exists(LOCAL_RUN_DIR):
        shutil.rmtree(LOCAL_RUN_DIR)
    shutil.copytree(DRIVE_RUN_DIR, LOCAL_RUN_DIR)

    os.makedirs(LOCAL_PREPROCESSED_DIR, exist_ok=True)
    for fn in ('test_x.h5', 'test_y.h5'):
        src = os.path.join(DRIVE_PREPROCESSED_DIR, fn)
        if not os.path.isfile(src):
            raise FileNotFoundError('Missing ' + src)
        shutil.copy2(src, os.path.join(LOCAL_PREPROCESSED_DIR, fn))

    RUN_DIR = LOCAL_RUN_DIR
    PREPROCESSED_DIR = LOCAL_PREPROCESSED_DIR
else:
    RUN_DIR = DRIVE_RUN_DIR
    PREPROCESSED_DIR = DRIVE_PREPROCESSED_DIR

print('RUN_DIR:', RUN_DIR)
print('PREPROCESSED_DIR:', PREPROCESSED_DIR)


In [ ]:
%cd {REPO_DIR}
!python -m pip install --upgrade pip
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install timm h5py tqdm huggingface_hub scikit-learn


Optional: if token env var is empty and model is gated, login once.


In [ ]:
# !huggingface-cli login


In [ ]:
import os
need = [
    os.path.join(PREPROCESSED_DIR, 'test_x.h5'),
    os.path.join(PREPROCESSED_DIR, 'test_y.h5'),
    os.path.join(RUN_DIR, 'model_best.pt'),
]
for p in need:
    print(('OK ' if os.path.exists(p) else 'MISSING '), p)

for p in ('temperature_fit.json', 'run_config.json'):
    q = os.path.join(RUN_DIR, p)
    print(('OK ' if os.path.exists(q) else 'MISSING '), q)


In [ ]:
%cd {REPO_DIR}
!python scripts/evaluate_virchow_preprocessed_test_colab.py \
  --preprocessed-dir "{PREPROCESSED_DIR}" \
  --run-dir "{RUN_DIR}"


Optional (slower): MC Dropout uncertainty pass.


In [ ]:
# %cd {REPO_DIR}
# !python scripts/evaluate_virchow_preprocessed_test_colab.py \
#   --preprocessed-dir "{PREPROCESSED_DIR}" \
#   --run-dir "{RUN_DIR}" \
#   --mc-samples 30


In [ ]:
import os, json
print('RUN_DIR:', RUN_DIR)
for name in ('test_metrics.json', 'test_metrics_detailed.json', 'test_predictions.npz'):
    p = os.path.join(RUN_DIR, name)
    print(('OK ' if os.path.exists(p) else 'MISSING '), p)

m = os.path.join(RUN_DIR, 'test_metrics.json')
if os.path.exists(m):
    with open(m, 'r', encoding='utf-8') as f:
        d = json.load(f)
    print('\nCompact test metrics:')
    for k, v in d.items():
        print(f'  {k}: {v}')
